In [1]:
# Library imports
import os, sys
import time
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
sys.path.append(os.path.abspath('..'))

# File imports
from env import InventoryEnv
from miscelaneous import read_instance
from policies.perfect_hindsight_policy import PerfectHindsightPolicy
from policies.exact_value_function_policy import ExactValueFunctionPolicy
from policies.auxiliaries.imitation_learning_model import ImitationLearningNet

In [2]:
def run_episode(env, policy, instance_file, output_dir):
    # policy is any expert with the BasePolicy interface (reset(instance)/act(state)) -
    # PerfectHindsightPolicy precomputes its whole action sequence in reset() and
    # replays it, while LinearProgrammingExactPolicy (LPE) decides live at each step, but
    # both are driven identically here since neither needs special-casing.

    num_warehouses = env.num_warehouses
    num_customers = env.num_customers
    capacity_distribution = env.capacity_distribution

    instance = read_instance(instance_file)
    env.set_instance(instance)
    policy.reset(instance)  # points env.get_state at the identity function - both experts need the raw state dict
    state, info = env.reset()

    done = False
    data = []

    while not done:

        action = policy.act(state)

        warehouses_distance = state["warehouses_distance"]
        warehouses_capacity = state["warehouses_capacity"]
        
        row = []

        for i in range(state['static_info']['num_warehouses']):
            row.append(warehouses_distance[i] / 212.13)
            row.append(warehouses_capacity[i] / state['static_info']['warehouses_initial_capacity'][i])

        row.append(action)

        data.append(row)

        state, reward, done, truncated, _ = env.step(action)

    #Save data in a csv file
    df = pd.DataFrame(data, columns=[f"warehouses_distance_{i}" for i in range(num_warehouses)] + [f"warehouses_capacity_{i}" for i in range(num_warehouses)] + ["action"])

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path / f"data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv", mode='a', header=False, index=False)

In [3]:
# Which expert policy the neural net will imitate - "perfect_hindsight" (the original
# oracle-with-future-knowledge expert) or "linear_programming_exact" (LPE, the same
# online, no-hindsight expert DecisionTreePolicy imitates - see
# train_decision_tree.ipynb). Each writes to its own output folder so the two datasets
# never mix; nn_model() below still reads from perfect_agent_data by default, so point
# it at linear_programming_exact_agent_data too if training on the LPE-imitation data.
EXPERT_OUTPUT_DIRS = {
    "perfect_hindsight": "imitation_learning_training/perfect_agent_data",
    "exact_value_function": "imitation_learning_training/perfect_agent_data"
}


def make_expert_policy(expert_name, env, num_warehouses):
    if expert_name == "perfect_hindsight":
        return PerfectHindsightPolicy(env)
    elif expert_name == "exact_value_function":
        return ExactValueFunctionPolicy(env, num_warehouses, env.num_customers, env.capacity_distribution)
    else:
        raise ValueError(f"Unknown expert_name: {expert_name!r} (expected 'perfect_hindsight' or 'linear_programming_exact')")


def export_data(num_warehouses, num_customers, capacity_distribution, expert_name="perfect_hindsight"):

    env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
    policy = make_expert_policy(expert_name, env, num_warehouses)

    num_instances = 5_000
    instances = [f'../instances/instances_train_imitation_learning/instances_seed_{i}.json' for i in range(20001,num_instances+20001)]

    output_dir = EXPERT_OUTPUT_DIRS[expert_name]

    # run_episode() below appends one instance's rows at a time (mode='a'), which is
    # how this family's dataset is meant to accumulate *within* this call - but without
    # clearing the file first, calling export_data() again for this same family (e.g.
    # rerunning this cell) would silently pile a second full 5,000-instance dataset on
    # top of the first instead of replacing it.
    output_path = Path(output_dir) / f"data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv"
    output_path.unlink(missing_ok=True)

    start_time = time.perf_counter()
    for instance in instances:
        run_episode(env, policy, instance, output_dir)
    data_collection_time_minutes = (time.perf_counter() - start_time) / 60

    return data_collection_time_minutes

In [4]:
EXPERT_NAME = "exact_value_function"  # or "linear_programming_exact"

num_customers = 50
data_collection_times = {}  # (num_warehouses, num_customers, capacity_distribution) -> minutes

for num_warehouses in [2, 3, 4, 5]:
    for capacity_distribution in ["uniform", "uneven"]:
        data_collection_time_minutes = export_data(num_warehouses, num_customers, capacity_distribution, expert_name=EXPERT_NAME)
        data_collection_times[(num_warehouses, num_customers, capacity_distribution)] = round(data_collection_time_minutes, 2)

In [5]:
def nn_model(num_warehouses, num_customers, capacity_distribution, input_dir="imitation_learning_training/perfect_agent_data"):

    # Read the data from the CSV file
    df = pd.read_csv(
        #f"imitation_learning_training/linear_programming_exact_agent_data/data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv",
        f"{input_dir}/data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv",
        header=None
    )
    # Rename columns to alternate distance/capacity
    columns = []
    for i in range(num_warehouses):
        columns.append(f"d_{i}")
        columns.append(f"c_{i}")
    columns.append("action")
    df.columns = columns

    # Prepare the input and output data
    X = df.drop(columns=["action"]).values.astype(np.float32)
    y = df["action"].values.astype(np.int64)

    # Split the data into training and testing sets (same split as before)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    # Further split the training data into train/validation - mirrors Keras' fit(..., validation_split=0.2)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

    X_train_t, y_train_t = torch.tensor(X_train), torch.tensor(y_train)
    X_val_t, y_val_t = torch.tensor(X_val), torch.tensor(y_val)
    X_test_t, y_test_t = torch.tensor(X_test), torch.tensor(y_test)

    # Deliberate reseed: fixes the network's weight init and (via the DataLoader
    # generator below) its per-epoch shuffle order, so a rerun trains the same model.
    SEED = 42
    torch.manual_seed(SEED)
    np.random.seed(SEED)

    # Build the neural network model - same architecture as the Keras version
    # (Dense 64 -> Dense 16 -> Dense num_warehouses, ReLU/ReLU), defined once in
    # policies/imitation_learning_model.py and shared with ImitationLearningPolicy so
    # training and inference can never drift apart
    model = ImitationLearningNet(num_warehouses)
    optimizer = torch.optim.Adam(model.parameters())
    # sparse_categorical_crossentropy on integer labels == CrossEntropyLoss on raw logits
    criterion = nn.CrossEntropyLoss()

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True,
        generator=torch.Generator().manual_seed(SEED),  # pins the per-epoch shuffle order too
    )

    # Train the model
    epochs = 25
    start_time = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_accuracy = (val_logits.argmax(dim=1) == y_val_t).float().mean().item()
        print(f"Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy:.4f}")
    training_time_minutes = (time.perf_counter() - start_time) / 60

    # Evaluate the model
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        accuracy = (test_logits.argmax(dim=1) == y_test_t).float().mean().item()
    print(f"Accuracy: {accuracy}")

    # Save the model
    torch.save(
        model.state_dict(),
        f"imitation_learning_training/imitation_learning_models/model_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.pt"
    )

    return training_time_minutes, accuracy

# NOTE: even with the above seeded, exact reproducibility across machines/time also
# needs pinned package versions (torch, etc. - requirements.txt already pins these) and,
# if this ever runs on GPU, torch.backends.cudnn.deterministic = True /
# torch.backends.cudnn.benchmark = False - not addressed here since training runs on CPU.

In [6]:
num_warehouses_options = [2, 3, 4, 5]
capacity_distribution_options = ["uniform", "uneven"]
num_customers = 50  # Fixed number of customers for training the neural network

accuracy_dict = {}
training_times = {}  # (num_warehouses, num_customers, capacity_distribution) -> minutes

for num_warehouses in num_warehouses_options:
    for capacity_distribution in capacity_distribution_options:
        training_time_minutes, accuracy = nn_model(num_warehouses, num_customers, capacity_distribution, input_dir=EXPERT_OUTPUT_DIRS[EXPERT_NAME])
        training_times[(num_warehouses, num_customers, capacity_distribution)] = round(training_time_minutes, 2)
        accuracy_dict[(num_warehouses, num_customers, capacity_distribution)] = accuracy

Epoch 1/25 - val_loss: 0.0498 - val_accuracy: 0.9865
Epoch 2/25 - val_loss: 0.0256 - val_accuracy: 0.9944
Epoch 3/25 - val_loss: 0.0196 - val_accuracy: 0.9940
Epoch 4/25 - val_loss: 0.0158 - val_accuracy: 0.9954
Epoch 5/25 - val_loss: 0.0203 - val_accuracy: 0.9907
Epoch 6/25 - val_loss: 0.0197 - val_accuracy: 0.9912
Epoch 7/25 - val_loss: 0.0129 - val_accuracy: 0.9951
Epoch 8/25 - val_loss: 0.0109 - val_accuracy: 0.9965
Epoch 9/25 - val_loss: 0.0124 - val_accuracy: 0.9947
Epoch 10/25 - val_loss: 0.0110 - val_accuracy: 0.9956
Epoch 11/25 - val_loss: 0.0119 - val_accuracy: 0.9946
Epoch 12/25 - val_loss: 0.0110 - val_accuracy: 0.9952
Epoch 13/25 - val_loss: 0.0181 - val_accuracy: 0.9914
Epoch 14/25 - val_loss: 0.0101 - val_accuracy: 0.9959
Epoch 15/25 - val_loss: 0.0180 - val_accuracy: 0.9917
Epoch 16/25 - val_loss: 0.0089 - val_accuracy: 0.9966
Epoch 17/25 - val_loss: 0.0101 - val_accuracy: 0.9956
Epoch 18/25 - val_loss: 0.0094 - val_accuracy: 0.9965
Epoch 19/25 - val_loss: 0.0099 - val_

In [7]:
# Merge the data-collection times (from the export_data sweep above) and the training
# times (from the nn_model sweep above) into one detailed table keyed by family, plus a
# version aggregated by num_customers - written once, after both sweeps have finished.
rows = []
for key, data_collection_time in data_collection_times.items():
    num_warehouses, num_customers, capacity_distribution = key
    rows.append({
        'num_warehouses': num_warehouses,
        'num_customers': num_customers,
        'capacity_distribution': capacity_distribution,
        'data_collection_time': data_collection_time,
        'function_fitting_time': training_times[key],
        'training_time': training_times[key] + data_collection_time,
        'accuracy': accuracy_dict[key]
    })

info_dir = 'imitation_learning_training'
os.makedirs(info_dir, exist_ok=True)

training_times_df = pd.DataFrame(rows)
training_times_df.to_csv(os.path.join(info_dir, 'imitation_learning_training_times.csv'), index=False, float_format='%.5f')